# 01 — Data Overview
Initial profiling of `blood_age_mega_raw.csv`: shape, dtypes, missingness, age distribution.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_raw, load_config, profile_missingness, profile_dtypes

sns.set_theme(style='whitegrid', font_scale=1.1)
config = load_config('../../config.yaml')
print('Config loaded.')

Config loaded.


## 1. Load raw data

In [2]:
df = load_raw(config=config)
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

Shape: 97683 rows x 60 columns


,SEQN,CYCLE,Age,CRP,LBDBANO,LBDEONO,LBDHDD,LBDLDL,LBDLYMNO,LBDMONO,...,LBXTR,LBXWBCSI,URXCRS,URXUCR,URXUMA,URXUMS,URDACT,LBXSCK,LBXNRBC,LBXMAGN
0,31127,2005-2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,31128,2005-2006,11.0,0.01,NaN,0.1,55.0,NaN,2.3,0.4,...,NaN,5.0,25724.0,291.0,82.1,82.1,NaN,NaN,NaN,NaN
2,31129,2005-2006,15.0,1.57,NaN,1.0,46.0,NaN,1.2,1.0,...,NaN,8.2,25459.0,288.0,17.8,17.8,NaN,NaN,NaN,NaN
3,31130,2005-2006,85.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,31131,2005-2006,44.0,2.44,NaN,NaN,39.0,49.0,1.9,0.4,...,86.0,5.3,17857.0,202.0,18.0,18.0,NaN,NaN,NaN,NaN


In [3]:
df.dtypes.to_frame('dtype')

,dtype
SEQN,object
CYCLE,object
Age,float64
CRP,float64
LBDBANO,float64
LBDEONO,float64
LBDHDD,float64
LBDLDL,float64
LBDLYMNO,float64
LBDMONO,float64


## 2. Missingness profile

In [4]:
miss = profile_missingness(df)
miss.head(20)

,column,missing_count,missing_pct
0,LBXMAGN,91359,93.53
1,LBXNRBC,79859,81.75
2,LBDLDL,68663,70.29
3,LBXTR,68278,69.90
4,LBXGLU,67575,69.18
5,LBDBANO,64188,65.71
6,LBXSCK,57236,58.59
7,CRP,40390,41.35
8,LBXSLDSI,39552,40.49
9,LBXSASSI,37703,38.60


In [5]:
fig, ax = plt.subplots(figsize=(12, 8))
# Only plot columns with >0% missing
plot_data = miss[miss['missing_pct'] > 0].copy()
colors = ['#e74c3c' if p > 60 else '#f39c12' if p > 30 else '#2ecc71' for p in plot_data['missing_pct']]
ax.barh(plot_data['column'], plot_data['missing_pct'], color=colors)
ax.set_xlabel('Missing %')
ax.set_title('Missingness by Column')
ax.axvline(x=60, color='red', linestyle='--', alpha=0.7, label='60% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('../../reports/figures/01_missingness_all_columns.png', dpi=150)
plt.show()
print('Saved: reports/figures/01_missingness_all_columns.png')

Saved: reports/figures/01_missingness_all_columns.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_13352\3088935333.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Age distribution & top-coding

In [6]:
ceiling = config['age']['top_code_ceiling']
topcoded_count = (df['Age'] == ceiling).sum()
print(f'Top-code ceiling: {ceiling}')
print(f'Rows at exactly {ceiling}: {topcoded_count} ({topcoded_count/len(df)*100:.1f}%)')
print(f'Max age in data: {df["Age"].max()}')

fig, ax = plt.subplots(figsize=(10, 5))
age_data = df['Age'].dropna()
ax.hist(age_data, bins=86, edgecolor='white', alpha=0.8)
ax.axvline(x=ceiling, color='red', linestyle='--', linewidth=2, label=f'Top-code at {ceiling}')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.set_title('Age Distribution with NHANES Top-Coding')
ax.legend()
plt.tight_layout()
plt.savefig('../../reports/figures/02_age_distribution.png', dpi=150)
plt.show()
print('Saved: reports/figures/02_age_distribution.png')

Top-code ceiling: 80
Rows at exactly 80: 3635 (3.7%)
Max age in data: 85.0
Saved: reports/figures/02_age_distribution.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_13352\2316423215.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Survey cycle breakdown

In [7]:
cycle_counts = df['CYCLE'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 5))
cycle_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_ylabel('Count')
ax.set_title('Participants per NHANES Cycle')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../../reports/figures/03_cycle_breakdown.png', dpi=150)
plt.show()

C:\Users\vinit\AppData\Local\Temp\ipykernel_13352\1632236520.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Column classification

In [8]:
target = config['columns']['target']
identifiers = config['columns']['identifiers']
phenoage = config['columns']['phenoage_core']
secondary = config['columns']['secondary']
drop_hm = config['columns']['drop_high_missing']
drop_dup = config['columns']['drop_duplicate']

classification = []
for col in df.columns:
    if col == target:
        role = 'TARGET'
    elif col in identifiers:
        role = 'IDENTIFIER'
    elif col in phenoage:
        role = 'PHENOAGE_CORE'
    elif col in secondary:
        role = 'SECONDARY'
    elif col in drop_hm:
        role = 'DROP_HIGH_MISSING'
    elif col in drop_dup:
        role = 'DROP_DUPLICATE'
    else:
        role = 'OTHER'
    miss_pct = df[col].isnull().mean() * 100
    classification.append({'column': col, 'role': role, 'missing_pct': round(miss_pct, 1)})

class_df = pd.DataFrame(classification)
class_df

,column,role,missing_pct
0,SEQN,IDENTIFIER,0.0
1,CYCLE,IDENTIFIER,0.0
2,Age,TARGET,3.8
3,CRP,PHENOAGE_CORE,41.3
4,LBDBANO,DROP_HIGH_MISSING,65.7
5,LBDEONO,OTHER,25.1
6,LBDHDD,SECONDARY,29.4
7,LBDLDL,SECONDARY,70.3
8,LBDLYMNO,OTHER,20.9
9,LBDMONO,OTHER,20.9


In [9]:
print('Column counts by role:')
print(class_df['role'].value_counts().to_string())

Column counts by role:
role
OTHER                38
PHENOAGE_CORE         9
SECONDARY             5
DROP_HIGH_MISSING     3
IDENTIFIER            2
DROP_DUPLICATE        2
TARGET                1


## 6. Summary statistics for candidate features

In [10]:
candidate_cols = phenoage + secondary + [target]
available = [c for c in candidate_cols if c in df.columns]
df[available].describe().round(2)

,LBXSAL,LBXSCR,LBXGLU,CRP,LBXLYPCT,LBXMCVSI,LBXRDW,LBXSAPSI,LBXWBCSI,LBXGH,LBDLDL,LBDHDD,LBXTR,LBXTC,Age
count,60204.00,60158.00,30108.00,57293.00,77250.00,77357.00,77357.00,60153.00,77355.00,61461.00,29020.00,68918.00,29405.00,68917.00,93933.00
mean,4.21,0.87,107.75,2.18,33.67,87.02,13.36,85.51,7.27,5.68,107.61,53.39,115.88,181.88,34.20
std,0.37,0.43,34.42,6.01,10.64,6.39,1.33,53.24,3.09,1.03,35.58,15.14,99.47,41.55,24.74
min,1.20,0.14,21.00,0.01,2.60,35.40,6.30,7.00,1.40,2.00,3.00,5.00,10.00,59.00,1.00
25%,4.00,0.69,93.00,0.15,26.30,83.40,12.50,58.00,5.70,5.20,82.00,43.00,64.00,152.00,12.00
50%,4.20,0.82,100.00,0.56,32.50,87.50,13.10,73.00,6.90,5.50,104.00,51.00,93.00,177.00,30.00
75%,4.40,0.99,109.00,1.97,39.80,91.20,13.80,92.00,8.40,5.80,129.00,62.00,139.00,207.00,56.00
max,5.60,17.80,584.00,246.86,94.50,125.30,37.80,907.00,400.00,17.80,375.00,226.00,4233.00,813.00,85.00


## 7. Duplicate column check

In [11]:
for dup, primary in [('LBXSCH', 'LBXTC'), ('LBXSGL', 'LBXGLU')]:
    mask = df[dup].notna() & df[primary].notna()
    if mask.sum() > 0:
        corr = df.loc[mask, dup].corr(df.loc[mask, primary])
        print(f'{dup} vs {primary}: r={corr:.4f} (n={mask.sum()} overlapping rows)')
    else:
        print(f'{dup} vs {primary}: no overlapping rows')

LBXSCH vs LBXTC: r=0.9888 (n=60135 overlapping rows)
LBXSGL vs LBXGLU: r=0.9849 (n=29510 overlapping rows)


## Summary
- **97,683 rows × 60 columns** in the raw data.
- **Age top-coding**: spike at 80 years (NHANES privacy cap).
- **6 columns** have >60% missingness and will be dropped.
- **2 duplicate alt-code columns** (LBXSCH, LBXSGL) will be dropped.
- **9 PhenoAge core + 5 secondary** biomarkers are the candidate features.
- Next notebook: missingness visualization + cleaning pipeline.